# Structured Knowledge Ingestion with Graph Databases (Neo4j)

This notebook introduces a critical step for building advanced Retrieval-Augmented Generation (RAG) systems: transforming unstructured text into structured knowledge graphs. Traditional RAG methods excel at semantic search, retrieving chunks of text that are *semantically related* to a query. However, they often fail when the answer requires understanding complex relationships—for example, "What is the relationship between Elon Musk's investments and his stated goals for SpaceX?" Simply retrieving paragraphs might provide context, but it won't explicitly map the connections (e.g., `(Musk)-[:INVESTED_IN]->(SpaceX)`).

By integrating a Large Language Model (LLM) with a graph database like Neo4j, we can perform sophisticated entity and relationship extraction. The `LLMGraphTransformer` uses the LLM's reasoning capabilities to read chunks of text and systematically identify key entities (nodes) and the types of connections (relationships) between them. This process moves our system from merely retrieving *information* to building a verifiable map of *knowledge*.

This structured approach is foundational for advanced RAG architectures, particularly those built with LangGraph. Instead of passing raw text chunks to the final LLM step, we can pass a graph query or a set of extracted facts. This allows downstream agents to perform multi-hop reasoning—following paths through the knowledge graph—to synthesize highly accurate and traceable answers that are far more robust than simple vector similarity search alone. By mastering this ingestion pipeline, you will learn how to build the foundational "brain" for an intelligent agent system.

### Learning Objectives

Upon completing this notebook, you will be able to:

*   **Understand Knowledge Graph Principles:** Explain the difference between unstructured text and structured knowledge graphs (nodes and relationships).
*   **Implement Text Chunking:** Use `RecursiveCharacterTextSplitter` to prepare large documents for processing while maintaining context.
*   **Perform Entity/Relationship Extraction:** Utilize `LLMGraphTransformer` to programmatically extract structured triples (Source $\rightarrow$ Relationship $\rightarrow$ Target) from text chunks using an LLM.
*   **Ingest into Graph Databases:** Store extracted graph data into a Neo4j database using LangChain's specialized graph loaders (`Neo4jGraph`).
*   **Create Vector Indexes on Graphs:** Build and manage vector indexes over existing structured nodes in Neo4j, ensuring that both semantic search and graph traversal are available for retrieval.


### Setup and Imports

This cell imports all necessary libraries and components for the advanced RAG pipeline. It includes tools for loading documents (PDFs), splitting text, interacting with OpenAI models (for chat and embeddings), generating graph structures from text (`LLMGraphTransformer`), and connecting to a Neo4j graph database for storage and vector operations.


In [2]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_neo4j import Neo4jGraph, Neo4jVector


In [3]:
load_dotenv()

True

### Model Initialization for Deterministic Extraction

This cell initializes the Language Model (LLM) and the embedding model. Setting `temperature=0` is crucial for deterministic behavior, ensuring consistent entity and relationship extraction necessary for structured data ingestion.


In [4]:
# temperature=0 ensures deterministic entity and relationship extraction
llm = ChatOpenAI(model="gpt-5-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


### Code Explanation

This cell initializes a `PyPDFLoader` to read the specified PDF file. The `.load()` method processes the entire document, yielding a list of LangChain `Document` objects, typically one per page. The subsequent loop iterates through these loaded pages and prints the character count (`len(p.page_content)`) for each page, allowing us to verify successful ingestion and measure content size.


In [5]:
# PyPDFLoader yields one Document per page
loader = PyPDFLoader("data/elon_musk.pdf") # Initialize the loader with the PDF file path
pages = loader.load() # Load all pages into a list of Document objects

for i, p in enumerate(pages): # Iterate through each loaded document (page)
    print(f"Page {i + 1}: {len(p.page_content)} chars") # Print the page number and the character count of its content


Page 1: 2343 chars
Page 2: 1192 chars


### Chunking Documents for Optimal Context

This cell uses `RecursiveCharacterTextSplitter` to break down large documents (`pages`) into smaller, manageable chunks. Smaller chunks provide the LLM with a more focused and tighter context, which is crucial for high-quality entity extraction and retrieval.


In [6]:
# smaller chunks give the LLM tighter context for entity extraction
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(pages)

print(f"{len(chunks)} chunks created")


14 chunks created


### Graph Initialization

This cell initializes the connection to a Neo4j graph database. The `Neo4jGraph` class handles the connection details (URI, username, password) using environment variables, making the graph accessible for subsequent data ingestion and knowledge representation.


In [7]:
graph = Neo4jGraph(
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
)

### Graph Transformer Initialization

This line initializes the `graph_transformer` object. This class is crucial for transforming raw graph structures (like those derived from knowledge graphs or semantic networks) into a format that can be effectively consumed and processed by the underlying Large Language Model (LLM).


In [8]:
graph_transformer = LLMGraphTransformer(llm=llm) # Initializes the Graph Transformer, passing the configured LLM instance to it.


### Graph Document Extraction

This cell converts the raw text chunks into structured graph documents. The `graph_transformer` (presumably a specialized model or class) processes the chunk list, extracting nodes (entities) and relationships between them. This is crucial for advanced RAG as it allows retrieval to return not just text, but also the underlying knowledge structure.


In [9]:
graph_docs = graph_transformer.convert_to_graph_documents(chunks)

print(f"{len(graph_docs)} graph documents extracted")

# spot-check the first extraction to verify the structure
print("Nodes:", [n.id for n in graph_docs[0].nodes])
print("Rels: ", [(r.source.id, r.type, r.target.id) for r in graph_docs[0].relationships])


14 graph documents extracted
Nodes: ['Elon Musk', 'June 28, 1971', 'Pretoria, South Africa', 'American', 'Entrepreneur', 'Engineer', "World'S Wealthiest Person"]
Rels:  [('Elon Musk', 'BORN_ON', 'June 28, 1971'), ('Elon Musk', 'BORN_IN', 'Pretoria, South Africa'), ('Elon Musk', 'HAS_NATIONALITY', 'American'), ('Elon Musk', 'HAS_OCCUPATION', 'Entrepreneur'), ('Elon Musk', 'HAS_OCCUPATION', 'Engineer'), ('Elon Musk', 'RECOGNISED_AS', "World'S Wealthiest Person")]


### Graph Storage and Indexing

This cell ingests the processed documents (`graph_docs`) into the Neo4j graph database. By setting `include_source=True`, it ensures that every generated entity node retains a link back to its original source document, which is crucial for traceability and subsequent retrieval steps using `Neo4jVector`.

*   **Function:** `add_graph_documents()`: Adds structured documents (containing entities and relationships) to the graph.


In [13]:
# include_source=True links each entity node back to its source Document node,
# which is required for Neo4jVector.from_existing_graph in the next cell
graph.add_graph_documents(
    graph_docs,
    include_source=True,
    baseEntityLabel=True
)
print("Graph stored in Neo4J")


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description="warn: feature deprecated with replacement. apoc.create.addLabels is deprecated. It is replaced by Cypher's dynamic labels; `SET n:$(labels)`..", position=<SummaryInputPosition line=1, column=257, offset=256>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 256, 'line': 1, 'column': 257}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MERGE (d:Document {id:$document.metadata.id}) SET d.text = $document.page_content SET d += $document.metadata WITH d UNWIND $data AS row MERGE (source:`__Entity__` {id: row.id}) SET source += row.properties MERGE (d)-[:MENTIONS]->(source) WITH source, row CALL apoc.create.addLabels( source, [row.type] 

Graph stored in Neo4J


### Vector Index Creation

This cell initializes a vector index, `Neo4jVector`, which is crucial for enabling semantic search. It connects to an existing Neo4j graph and creates the necessary indexing structure over the stored document nodes using the provided embeddings.


In [14]:
# create a vector index over the Document nodes stored above
vector_index = Neo4jVector.from_existing_graph(
    embedding=embeddings,
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
    index_name="elon_musk_chunks",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding",
)
print("Vector index created")


Vector index created


### Verification of Graph Structure

This cell queries the Neo4j graph database to verify that the ingestion process successfully created nodes and relationships. It counts the occurrences of different node labels and relationship types, providing a structural check before proceeding with advanced RAG operations.


In [15]:
# verify what landed in Neo4J
# Query for all unique node labels and count how many nodes belong to each label.
node_counts = graph.query(
    "MATCH (n) RETURN labels(n) AS label, count(n) AS count ORDER BY count DESC"
)
# Query for all unique relationship types and count the total number of relationships of that type.
rel_counts = graph.query(
    "MATCH ()-[r]->() RETURN type(r) AS type, count(r) AS count ORDER BY count DESC"
)
print("Nodes:")
# Iterate through the results to print node label counts.
for r in node_counts:
    print(" ", r)
print("Relationships:")
# Iterate through the results to print relationship type counts.
for r in rel_counts:
    print(" ", r)


Nodes:
  {'label': ['__Entity__', 'Person'], 'count': 24}
  {'label': ['Document'], 'count': 14}
  {'label': ['__Entity__', 'Location'], 'count': 12}
  {'label': ['__Entity__', 'Organization'], 'count': 11}
  {'label': ['__Entity__', 'Product'], 'count': 10}
  {'label': ['__Entity__', 'Occupation'], 'count': 5}
  {'label': ['__Entity__', 'Date'], 'count': 4}
  {'label': ['__Entity__', 'Nationality'], 'count': 3}
  {'label': ['__Entity__', 'Year'], 'count': 3}
  {'label': ['__Entity__', 'Concept'], 'count': 3}
  {'label': ['__Entity__', 'Company'], 'count': 2}
  {'label': ['__Entity__', 'Service'], 'count': 2}
  {'label': ['__Entity__', 'Recognition'], 'count': 1}
  {'label': ['__Entity__', 'Software'], 'count': 1}
  {'label': ['__Entity__', 'Amount', 'Monetaryamount'], 'count': 1}
  {'label': ['__Entity__', 'Goal'], 'count': 1}
  {'label': ['__Entity__', 'Research'], 'count': 1}
  {'label': ['__Entity__', 'Thing'], 'count': 1}
Relationships:
  {'type': 'MENTIONS', 'count': 115}
  {'typ